# EDA

In [1]:
import sys, os
sys.path.append(os.path.abspath(os.path.join("..", "src")))

import pandas as pd
pd.set_option("display.max_columns", None)

from ingesta import CargarDatos
from gestor import GestorPartidos
from eda import ProceEDA

# Ingesta de datos

In [2]:
os.chdir("..")  # trabajar desde la raíz del proyecto para rutas relativas (data/...)

cargador = CargarDatos()

if os.path.exists(cargador.ruta_raw):
    df_crudo = cargador.cargar_raw_existente()
else:
    df_crudo = cargador.ejecutar_pipeline_ingesta()

df_crudo.head()

Partidos descargados en total: 49505
Partidos de Copa Mundial encontrados: 1062
Los datos son válidos.
Archivo guardado en data/raw/partidos-mundial.csv


,id_partido,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1,1930-07-13,Belgium,United States,0.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
1,2,1930-07-13,France,Mexico,4.0,1.0,FIFA World Cup,Montevideo,Uruguay,True
2,3,1930-07-14,Brazil,Yugoslavia,1.0,2.0,FIFA World Cup,Montevideo,Uruguay,True
3,4,1930-07-14,Peru,Romania,1.0,3.0,FIFA World Cup,Montevideo,Uruguay,True
4,5,1930-07-15,Argentina,France,1.0,0.0,FIFA World Cup,Montevideo,Uruguay,True


In [4]:
df_crudo.info()

<class 'pandas.DataFrame'>
RangeIndex: 1062 entries, 0 to 1061
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id_partido  1062 non-null   int64  
 1   date        1062 non-null   str    
 2   home_team   1062 non-null   str    
 3   away_team   1062 non-null   str    
 4   home_score  1062 non-null   float64
 5   away_score  1062 non-null   float64
 6   tournament  1062 non-null   str    
 7   city        1062 non-null   str    
 8   country     1062 non-null   str    
 9   neutral     1062 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(6)
memory usage: 75.8 KB


# Limpieza de datos y columnas derivadas

In [5]:
eda = ProceEDA(df_crudo)
df_limpio = eda.limpieza_datos()
df_procesado = eda.generar_columnas_derivadas()
df_procesado.head()

Limpieza completada: 1062 -> 1062 filas
Columnas agregadas: anio, total_goles, diferencia_goles, ganador


,id_partido,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,anio,total_goles,diferencia_goles,ganador
0,1,1930-07-13,Belgium,United States,0,3,FIFA World Cup,Montevideo,Uruguay,True,1930,3,3,Visitante
1,2,1930-07-13,France,Mexico,4,1,FIFA World Cup,Montevideo,Uruguay,True,1930,5,3,Local
2,3,1930-07-14,Brazil,Yugoslavia,1,2,FIFA World Cup,Montevideo,Uruguay,True,1930,3,1,Visitante
3,4,1930-07-14,Peru,Romania,1,3,FIFA World Cup,Montevideo,Uruguay,True,1930,4,2,Visitante
4,5,1930-07-15,Argentina,France,1,0,FIFA World Cup,Montevideo,Uruguay,True,1930,1,1,Local


In [6]:
# Datos procesados guardados en un csv
rutas = cargador.guardar_procesado(df_procesado, nombre_archivo="partidos-mundial-procesado")
rutas

Archivo guardado en data/processed/partidos-mundial-procesado.csv


# Estadisticas

In [7]:
eda.resumen_descriptivo()

,home_score,away_score,total_goles,diferencia_goles
count,1062.000000,1062.000000,1062.000000,1062.000000
mean,1.582863,1.246704,2.829567,1.500000
std,1.494636,1.286251,1.912347,1.407366
min,0.000000,0.000000,0.000000,0.000000
25%,0.250000,0.000000,1.000000,1.000000
50%,1.000000,1.000000,3.000000,1.000000
75%,2.000000,2.000000,4.000000,2.000000
max,10.000000,8.000000,12.000000,9.000000


# Matriz de correlacion

In [8]:
matriz_corr = eda.matriz_correlacion()
matriz_corr

,home_score,away_score,total_goles,diferencia_goles,anio
home_score,1.000000,-0.060159,0.741108,0.557618,-0.124456
away_score,-0.060159,1.000000,0.625585,0.252519,-0.154108
total_goles,0.741108,0.625585,1.000000,0.605664,-0.200925
diferencia_goles,0.557618,0.252519,0.605664,1.000000,-0.139392
anio,-0.124456,-0.154108,-0.200925,-0.139392,1.000000


# Outliers en goles totales por partido

In [9]:
outliers = eda.detectar_outliers("total_goles")
print(f"Partidos con goleadas atípicas (outliers): {len(outliers)}")
outliers[["date", "home_team", "away_team", "home_score", "away_score", "total_goles"]]

Partidos con goleadas atípicas (outliers): 10


,date,home_team,away_team,home_score,away_score,total_goles
9,1930-07-19,Argentina,Mexico,6,3,9
36,1938-06-05,Brazil,Poland,6,5,11
81,1954-06-17,Hungary,South Korea,9,0,9
88,1954-06-20,Germany,Hungary,3,8,11
91,1954-06-23,Germany,Turkey,7,2,9
94,1954-06-26,Switzerland,Austria,5,7,12
105,1958-06-08,France,Paraguay,7,3,10
134,1958-06-28,France,Germany,6,3,9
243,1974-06-18,Yugoslavia,DR Congo,9,0,9
311,1982-06-15,Hungary,El Salvador,10,1,11


# Goles por edicion de mundial

In [10]:
goles_edicion = eda.goles_por_edicion()
goles_edicion

,anio,partidos,total_goles,promedio_goles
0,1930,18,70,3.89
1,1934,17,70,4.12
2,1938,18,84,4.67
3,1950,22,88,4.00
4,1954,26,140,5.38
5,1958,35,126,3.60
6,1962,32,89,2.78
7,1966,32,89,2.78
8,1970,32,95,2.97
9,1974,38,97,2.55


# ¿En que mundial se hicieron mas goles por partidos?

In [11]:
fila_max = goles_edicion.loc[goles_edicion["promedio_goles"].idxmax()]
print(f"La edición con más goles por partido en promedio fue {int(fila_max['anio'])} "
      f"con {fila_max['promedio_goles']:.2f} goles/partido.")

La edición con más goles por partido en promedio fue 1954 con 5.38 goles/partido.


# Resultados historicos por equipo

In [12]:
resultados_equipo = eda.resultados_por_equipo()
resultados_equipo.head(15)

,equipo,partidos_jugados,victorias,empates,derrotas,goles_favor,goles_contra,diferencia_gol
0,Brazil,119,79,20,20,247,112,135
1,Germany,116,70,22,24,243,135,108
2,France,79,45,14,20,152,87,65
3,Argentina,93,52,17,24,166,106,60
4,Italy,83,45,21,17,128,77,51
5,Netherlands,59,32,16,11,107,57,50
6,Spain,73,36,18,19,119,76,43
7,England,79,36,23,20,115,73,42
8,Hungary,32,15,3,14,87,57,30
9,Portugal,40,19,8,13,69,44,25


# ¿Que seleccion tiene la mejor diferncia de goles?

In [13]:
mejor_equipo = resultados_equipo.iloc[0]
print(f"{mejor_equipo['equipo']} tiene la mejor diferencia de gol histórica en Mundiales: "
      f"{mejor_equipo['diferencia_gol']} ({mejor_equipo['goles_favor']} a favor, "
      f"{mejor_equipo['goles_contra']} en contra, en {mejor_equipo['partidos_jugados']} partidos)."
)

Brazil tiene la mejor diferencia de gol histórica en Mundiales: 135 (247 a favor, 112 en contra, en 119 partidos).


# ¿El pais sede gana mas?

In [14]:
df_anfitrion = eda.anfitrion_gana_mas()
df_anfitrion

,anio,pais_sede,partidos_jugados_por_sede,victorias_sede,pct_victorias_sede
0,1930,Uruguay,4,4,100.00
1,1934,Italy,5,4,80.00
2,1938,France,2,1,50.00
3,1950,Brazil,6,4,66.67
4,1954,Switzerland,4,2,50.00
5,1958,Sweden,6,4,66.67
6,1962,Chile,6,4,66.67
7,1966,England,6,5,83.33
8,1970,Mexico,4,2,50.00
9,1974,Germany,7,6,85.71


In [17]:
promedio_anfitrion = df_anfitrion["pct_victorias_sede"].mean()
print(f"En promedio, el país anfitrión gana el {promedio_anfitrion:.1f}% de sus partidos, "
      f"frente al 45.7% de victorias históricas del local en general (ver GestorPartidos.ventaja_local()).")

En promedio, el país anfitrión gana el 58.5% de sus partidos, frente al 45.7% de victorias históricas del local en general (ver GestorPartidos.ventaja_local()).


# Consultas con GestorPartidos

In [18]:
gestor = GestorPartidos(df_procesado)

print("Ediciones disponibles:", gestor.get_ediciones_disponibles())
print("\nVentaja de local (histórico):")
gestor.ventaja_local()

Ediciones disponibles: [np.int32(1930), np.int32(1934), np.int32(1938), np.int32(1950), np.int32(1954), np.int32(1958), np.int32(1962), np.int32(1966), np.int32(1970), np.int32(1974), np.int32(1978), np.int32(1982), np.int32(1986), np.int32(1990), np.int32(1994), np.int32(1998), np.int32(2002), np.int32(2006), np.int32(2010), np.int32(2014), np.int32(2018), np.int32(2022), np.int32(2026)]

Ventaja de local (histórico):


{'total_partidos': 1062,
 'pct_victorias_local': np.float64(45.76),
 'pct_victorias_visitante': np.float64(31.83),
 'pct_empates': np.float64(22.41)}

In [19]:
# Ejemplo: enfrentamientos históricos entre dos selecciones
gestor.get_enfrentamientos("Argentina", "Germany")[
    ["date", "home_team", "away_team", "home_score", "away_score", "anio"]
]

,date,home_team,away_team,home_score,away_score,anio
101,1958-06-08,Argentina,Germany,1,3,1958
180,1966-07-16,Argentina,Germany,0,0,1966
411,1986-06-29,Argentina,Germany,3,2,1986
463,1990-07-08,Germany,Argentina,1,0,1990
700,2006-06-30,Germany,Argentina,1,1,2006
766,2010-07-03,Argentina,Germany,0,4,2010
835,2014-07-13,Germany,Argentina,1,0,2014
